In [39]:
from models import ( 
    KNNClassifier, 
    DecisionTree, 
    RandomForest, 
    ArtificialNeuralNetwork, 
    LogisticRegression
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from matplotlib import pyplot as plt

import numpy as np
import pandas as pd
import joblib

X_train = pd.read_csv(r"data\cleaned\X_train_scaled.csv", index_col=0)
X_val = pd.read_csv(r"data\cleaned\X_val_scaled.csv", index_col=0)
y_train = pd.read_csv(r"data\cleaned\y_train.csv", index_col=0)
y_val = pd.read_csv(r"data\cleaned\y_val.csv", index_col=0)

base_models = {
    "decision-tree": joblib.load("output/dtl1.pkl"),
    "random-forest": joblib.load("output/rf1.pkl"),
    "knn": joblib.load("output/knn_model_diy.pkl"),
    "logistic-regression": joblib.load("output/logistic_model_diy.pkl"),
    "ann": joblib.load("output/ann_relu_model.pkl")
}

knn_config = joblib.load("output/knn_model_diy_config.pkl")
knn_metric = knn_config["metric"]

In [40]:
def tree_features(node):
    if not isinstance(node, dict):
        return set()

    return (
        {node["feature"]}
        | tree_features(node["children"]["left"])
        | tree_features(node["children"]["right"])
    )


used_features = tree_features(base_models["decision-tree"].tree)

for tree in base_models["random-forest"].forest:
    used_features |= tree_features(tree.tree)

unexpected = used_features - set(X_train.columns)

print("Fitur pohon di luar fitur training:", unexpected)

Fitur pohon di luar fitur training: {'Curricular units 1st sem (credited)', 'Curricular units 2nd sem (credited)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (without evaluations)'}


In [41]:
base_models["decision-tree"] = DecisionTree(max_depth=1)
base_models["decision-tree"].fit(X_train, y_train)

base_models["random-forest"] = RandomForest()
base_models["random-forest"].fit(X_train, y_train)

Tree 10 of 10...

In [42]:
def predict_model(X):
    return pd.DataFrame(
        {
            "decision-tree" : base_models["decision-tree"].predict(X),
            "random-forest" : base_models["random-forest"].predict(X),
            "knn" : base_models["knn"].predict(X, distance_metric=knn_metric, batch_size=64),
            "logistic-regression" : base_models["logistic-regression"].predict(X),
            "ann" : base_models["ann"].predict(X.to_numpy(dtype=float))
        }, 
        index=X.index
    )

val_predictions = predict_model(X_val)

val_predictions.head()

,decision-tree,random-forest,knn,logistic-regression,ann
1065,0,0,0,0,0
2990,2,2,2,2,2
2808,2,2,2,2,2
172,2,2,2,2,1
2712,2,2,2,2,0


## **Normal Vote**

In [43]:
def vote(predictions):
    return (predictions.mode(axis=1).iloc[:, 0].to_numpy(dtype=int))

std_vote_pred = vote(val_predictions)

print("Classification Report - Hard voting")
print(classification_report(
    np.asarray(y_val).ravel(),
    std_vote_pred,
    labels=[0, 1, 2],
    target_names=["Dropout", "Enrolled", "Graduate"],
    digits=4,
    zero_division=0,
))


Classification Report - Hard voting
              precision    recall  f1-score   support

     Dropout     0.7385    0.7711    0.7544       249
    Enrolled     0.7619    0.1151    0.2000       139
    Graduate     0.7363    0.9404    0.8259       386

    accuracy                         0.7377       774
   macro avg     0.7456    0.6089    0.5935       774
weighted avg     0.7416    0.7377    0.6905       774



In [44]:
results = []

for name in val_predictions.columns:
    pred = val_predictions[name].to_numpy()

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_val, pred),
        "macro_f1": f1_score(
            y_val, pred, average="macro", zero_division=0
        ),
    })

results.append({
    "model": "ensemble",
    "accuracy": accuracy_score(y_val, std_vote_pred),
    "macro_f1": f1_score(
        y_val,
        std_vote_pred,
        average="macro",
        zero_division=0,
    ),
})

comparison = pd.DataFrame(results).sort_values(
    "macro_f1", ascending=False
)

display(comparison)


,model,accuracy,macro_f1
3,logistic-regression,0.753230,0.666518
2,knn,0.741602,0.636303
5,ensemble,0.737726,0.593453
4,ann,0.631783,0.551358
1,random-forest,0.717054,0.520015
0,decision-tree,0.704134,0.507326


## **Weighted Vote**

In [45]:
from sklearn.metrics import accuracy_score

accuracies = pd.Series({
    name: accuracy_score(y_val, val_predictions[name]) for name in val_predictions.columns
})

weights = accuracies / accuracies.sum()

display(pd.DataFrame({
    "validation_accuracy": accuracies,
    "weight": weights,
}).sort_values("weight", ascending=False))

,validation_accuracy,weight
logistic-regression,0.753230,0.212309
knn,0.741602,0.209031
random-forest,0.717054,0.202112
decision-tree,0.704134,0.198471
ann,0.631783,0.178077


In [46]:
def weighted_vote(predictions, weights, classes):
    weights = weights.loc[predictions.columns].to_numpy(dtype=float)
    classes = np.asarray(classes)
    pred_array = predictions.to_numpy()

    scores = np.column_stack([ (pred_array == label) @ weights for label in classes])

    winners = classes[np.argmax(scores, axis=1)]
    
    return winners, pd.DataFrame(
        scores,
        index=predictions.index,
        columns=classes,
    )

In [47]:
classes = np.sort(y_train["Target"].unique())
weight_pred, weight_score = weighted_vote(val_predictions, weights, classes)

print("Classification Report - Weighted hard voting")
print(classification_report(
    np.asarray(y_val).ravel(),
    weight_pred,
    labels=[0, 1, 2],
    target_names=["Dropout", "Enrolled", "Graduate"],
    digits=4,
    zero_division=0,
))

Classification Report - Weighted hard voting
              precision    recall  f1-score   support

     Dropout     0.7689    0.7349    0.7515       249
    Enrolled     0.6207    0.1295    0.2143       139
    Graduate     0.7219    0.9482    0.8197       386

    accuracy                         0.7326       774
   macro avg     0.7038    0.6042    0.5952       774
weighted avg     0.7188    0.7326    0.6891       774



## **Perbandingan Hasil Voting**

In [48]:
comparison = pd.DataFrame([
    {
        "method": "Hard voting",
        "accuracy": accuracy_score(y_val, std_vote_pred),
        "macro_f1": f1_score(
            y_val, std_vote_pred, average="macro", zero_division=0
        ),
    },
    {
        "method": "Weighted hard voting",
        "accuracy": accuracy_score(y_val, weight_pred),
        "macro_f1": f1_score(
            y_val, weight_pred, average="macro", zero_division=0
        ),
    }
])

display(comparison)

,method,accuracy,macro_f1
0,Hard voting,0.737726,0.593453
1,Weighted hard voting,0.732558,0.595178


## **Ensemble dengan Library scikit-learn**


In [49]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
from sklearn.neural_network import MLPClassifier

y_train_library = np.asarray(y_train).ravel()
y_val_library = np.asarray(y_val).ravel()

library_estimators = [
    ("decision-tree", DecisionTreeClassifier()),
    ("random-forest", RandomForestClassifier()),
    ("knn", KNeighborsClassifier()),
    ("logistic-regression", SklearnLogisticRegression(C=0.1, class_weight=None)),
    ("ann", MLPClassifier()),
]

## **Library: Hard Voting**

In [50]:
library_hard = VotingClassifier(estimators=library_estimators, voting="hard")
library_hard.fit(X_train, y_train_library)
library_hard_pred = library_hard.predict(X_val)

print("Classification Report - Library Hard Voting")
print(classification_report(
    y_val_library, library_hard_pred, labels=[0, 1, 2],
    target_names=["Dropout", "Enrolled", "Graduate"],
    digits=4, zero_division=0,
))

Classification Report - Library Hard Voting
              precision    recall  f1-score   support

     Dropout     0.7957    0.7349    0.7641       249
    Enrolled     0.4835    0.3165    0.3826       139
    Graduate     0.7881    0.9249    0.8510       386

    accuracy                         0.7545       774
   macro avg     0.6891    0.6588    0.6659       774
weighted avg     0.7358    0.7545    0.7389       774



c:\Users\uhamz\miniconda3\envs\advanced_model\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


## **Library: Weighted Hard Voting**

In [ ]:
# Gunakan bobot DIY sebelumnya, sesuai urutan nama estimator.
sklearn_weight_vote = VotingClassifier(
    estimators=library_estimators,
    voting="hard",
    weights=weights.loc[[name for name, _ in library_estimators]].to_list(),
)
sklearn_weight_vote.fit(X_train, y_train_library)
sklearn_weight_vote_pred = sklearn_weight_vote.predict(X_val)

print("Classification Report - Library Weighted Hard Voting")
print(classification_report(
    y_val_library, sklearn_weight_vote_pred, labels=[0, 1, 2],
    target_names=["Dropout", "Enrolled", "Graduate"],
    digits=4, zero_division=0,
))

## **Perbandingan Ensemble DIY dan Library**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

diy_hard_val_pred = vote(val_predictions)
diy_weighted_val_pred, _ = weighted_vote(val_predictions, weights, classes)
ensemble_comparison = pd.DataFrame([
    {
        "method": method,
        "accuracy": accuracy_score(y_val_library, predictions),
        "macro_precision": precision_score(y_val_library, predictions, average="macro", zero_division=0),
        "macro_recall": recall_score(y_val_library, predictions, average="macro", zero_division=0),
        "macro_f1": f1_score(y_val_library, predictions, average="macro", zero_division=0),
    }
    for method, predictions in [
        ("DIY Hard voting", diy_hard_val_pred),
        ("DIY Weighted hard voting", diy_weighted_val_pred),
        ("Library Hard voting", library_hard_pred),
        ("Library Weighted hard voting", sklearn_weight_vote_pred),
    ]
])
display(ensemble_comparison.round(4))

In [ ]:
test_df = pd.read_csv("data/cleaned/test_df.csv", index_col=0)
X_test = test_df.loc[:, X_train.columns]

test_predictions = predict_model(X_test)
weight_pred, test_scores = weighted_vote(test_predictions,weights,classes)
weight_pred = np.asarray(weight_pred)

target_labels = pd.Series(weight_pred).map({
    0: "Dropout",
    1: "Enrolled",
    2: "Graduate",
})

submission = pd.DataFrame({
    "Student_ID": test_df["Student_ID"].to_numpy(),
    "Target": target_labels.to_numpy(),
})

submission.head()

,Student_ID,Target
0,2701,Dropout
1,1142,Graduate
2,3538,Dropout
3,3000,Dropout
4,476,Graduate


In [ ]:
submission.to_csv("output/csv/submission_ensemble_weighted.csv", index=False, )